# Lab 24 · Use your gateway's models and tools in GitHub Copilot

Publish `gpt-6.1-sol` for GitHub Copilot through your AI Gateway, run the Copilot CLI and the Copilot SDK on it, then
give both the Microsoft Learn tools from an MCP server on the same gateway. Every model call and every tool call uses
one runtime key, so you can cut Copilot off by suspending it. This notebook is the code track of the
[lab guide](README.md): it deploys [infra/copilot-model.bicep](infra/copilot-model.bicep), runs the CLI in
non-interactive mode, and runs the guide's Python sample, [infra/copilot_sdk.py](infra/copilot_sdk.py), unchanged.
VS Code and the GitHub Copilot app have no code to run: connect them with the guide.

**Publish gpt-6.1-sol → Run the Copilot CLI and SDK on it → Add the Learn tools → Suspend the key**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| [infra/copilot-model.bicep](infra/copilot-model.bicep) | Deploys `gpt-6.1-sol` in Foundry, registers it on the gateway with a token limit of 100,000 tokens per minute for each caller, publishes the alias `gpt-6.1-sol`, and creates the runtime key `github-copilot`. |
| Model endpoint `<gateway-url>/default/models/openai/v1` | The OpenAI-compatible endpoint. Copilot calls its Responses API, and sends the key as a bearer token. |
| MCP endpoint `<gateway-url>/default/toolservers/learn/mcp` | The `learn` MCP server from lab 04, in front of the public Microsoft Learn MCP server. Copilot sends the same key in the `api-key` header. |
| GitHub Copilot CLI | Runs with `copilot -p` and four `COPILOT_PROVIDER_*` settings, so its agent loop runs on your machine and its model calls go to the gateway. |
| [infra/copilot_sdk.py](infra/copilot_sdk.py) | A Copilot SDK session with the gateway as its `provider`, and the Learn tools when `AI_GATEWAY_MCP_URL` is set. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

![Four GitHub Copilot surfaces send requests with one runtime key to the gpt-6.1-sol alias and the learn MCP server on the gateway](../assets/diagrams/github-copilot-byok.svg)

## Prerequisites

- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`, with
  Contributor on the gateway's resource group and on the Foundry account, so the notebook can deploy a model.
- Quota for `gpt-6.1-sol` (version `2026-09-29`, Global Standard) in the Foundry account behind your gateway. The
  notebook deploys it with 100,000 tokens per minute. Model inference is billed per use.
- [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) has run, so the gateway has the
  Foundry provider `foundry-models`.
- Python 3.11 or later with Jupyter, for the Copilot SDK, which brings its own Copilot runtime.
- Optional: the [GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/set-up/install-copilot-cli), tested
  with 1.0.106, on the kernel's `PATH`. Without it, the notebook skips the CLI steps.

The notebook gives the CLI and the SDK their own temporary `COPILOT_HOME`, so they don't read or change your Copilot
settings, sessions, or MCP servers. It also sets `COPILOT_OFFLINE=true`, which turns off the GitHub sign-in,
telemetry, web tools, and the GitHub MCP server: each model call and each MCP tool call goes to your gateway.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it reuses what exists.

In [ ]:
%pip install -q -r infra/requirements.txt openai azure-identity requests

### 0. Initialize the lab

The defaults match notebooks 01 and 02, so `utils.Lab` finds the gateway and its Foundry account without edits. If
`pip` upgraded a package that the kernel had already loaded, restart the kernel before you run this cell.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from collections import Counter
from pathlib import Path

import requests
from IPython.display import display
from openai import OpenAI

sys.path.insert(0, "../shared")   # helpers and brand styles that every lab notebook shares
import brand
import utils

lab = utils.Lab("24-github-copilot-byok")
model       = "gpt-6.1-sol"      # the Foundry deployment, the backend model, and the alias
provider    = "foundry-models"   # the gateway's Foundry provider from lab 02
key_name    = "github-copilot"   # the runtime key that every Copilot surface uses
server_name = "learn"            # the Microsoft Learn MCP server from lab 04

### 1. Publish gpt-6.1-sol for Copilot

The template deploys `gpt-6.1-sol` in the Foundry account, registers it on the provider with a token limit of
100,000 tokens per minute for each caller, publishes the alias `gpt-6.1-sol`, and creates the key `github-copilot`.
Coding agents send large prompts, which is why the limit is that high. The cell records what already existed, so
**Clean up** deletes only what this notebook created. The deployment takes two to three minutes.

In [ ]:
lab.require(f"{lab.workspace_id}/modelProviders/{provider}",
            "Run notebook 02 first, or set provider to the name of your Foundry provider.")
created_alias = lab.arm("GET", f"{lab.workspace_id}/aliases/{model}") is None
created_key   = lab.arm("GET", f"{lab.gateway_id}/apiKeys/{key_name}") is None

outputs = lab.deploy("infra/copilot-model.bicep", gatewayName=lab.gateway_name, foundryAccountId=lab.foundry_id,
                     modelName=model, providerName=provider, apiKeyName=key_name)
base_url = outputs["modelEndpoint"]
api_key, _ = lab.api_key(key_name)
display(brand.key(key_name, api_key, created_key))

### 2. Test the endpoint the way Copilot calls it

Copilot sends a Responses API request to the base URL, with the alias as `model` and the key as a bearer token. The
OpenAI SDK does the same. A new key and a new alias can take a minute to work, so the call retries on `401` and
`404`.

In [ ]:
client = OpenAI(base_url=base_url, api_key=api_key)
response = utils.retry(lambda: client.responses.create(model=model, input="In one sentence, what is a code review for?"))
display(brand.reply(response.output_text, "Responses API through the gateway", model=response.model,
                    tokens=(response.usage.input_tokens, response.usage.output_tokens), sub=f"POST {base_url}/responses"))

### 3. Create the practice project

The CLI works on a small project with two deliberate problems: `add` subtracts, and `divide` doesn't guard against
zero. The cell writes `~/contoso-calc/calc.py`, the same project that the guide uses, unless it already exists.

In [ ]:
project = Path.home() / "contoso-calc"
created_project = not project.exists()
if created_project:
    project.mkdir(parents=True)
    (project / "calc.py").write_text('''"""Tiny calculator used to try GitHub Copilot with a model from Azure AI Gateway."""


def add(a: float, b: float) -> float:
    return a - b


def divide(a: float, b: float) -> float:
    return a / b


if __name__ == "__main__":
    print(add(2, 3), divide(10, 2))
''')
display(brand.note(f"{'Created' if created_project else 'Reusing'} the practice project", "ok", project))

### 4. Point the Copilot CLI at the gateway

Four settings switch the CLI to your own model: the base URL, the Responses API, the key, and the model. The CLI
can then run without a GitHub sign-in. `run_copilot` runs one prompt with `copilot -p` in the practice project, and
reads what happened from the CLI's JSON events: the answer, the model calls, the tools it called, and any error.
`--secret-env-vars` keeps the key out of the agent's shell commands and output.

Ask a quick question. `premiumRequests` stays at `0`: with your own model, Copilot doesn't use GitHub's.

In [ ]:
copilot = shutil.which("copilot")
copilot_home = Path(tempfile.mkdtemp(prefix="copilot-home-"))   # keeps your own Copilot settings out of the runs
cli_settings = {
    "COPILOT_PROVIDER_BASE_URL": base_url,
    "COPILOT_PROVIDER_WIRE_API": "responses",
    "COPILOT_PROVIDER_API_KEY":  api_key,
    "COPILOT_MODEL":             model,
    "COPILOT_OFFLINE":           "true",
    "COPILOT_HOME":              str(copilot_home),
    "AI_GATEWAY_API_KEY":        api_key,       # the MCP configuration in step 8 reads the key from here
}


def run_copilot(prompt, *flags):
    """Run copilot -p in the practice project, and return what happened, read from its JSON events."""
    status = brand.live(brand.note(f"Copilot CLI: {prompt[:100]}", "wait"))
    start = time.time()
    done = subprocess.run([copilot, "-p", prompt, *flags, "--output-format", "json",
                           "--secret-env-vars=COPILOT_PROVIDER_API_KEY"],
                          cwd=project, env={**os.environ, **cli_settings}, capture_output=True, text=True, timeout=900)
    events = [json.loads(line) for line in done.stdout.splitlines() if line.startswith("{")]

    def data(kind):
        return [event.get("data", event) for event in events if event["type"] == kind]

    answers = [m for m in data("assistant.message") if m.get("phase") == "final_answer"] or data("assistant.message")
    run = {
        "exit_code":  done.returncode,
        "answer":     answers[-1]["content"] if answers else "",
        "model":      answers[-1].get("model", model) if answers else model,
        "calls":      len(data("model.call_start")),
        "failures":   [failure["statusCode"] for failure in data("model.call_failure")],
        "tools":      [tool["toolName"] for tool in data("tool.execution_start")],
        "error":      next((error["message"] for error in data("session.error")), done.stderr.strip()[-300:]),
        "premium":    next((result["usage"].get("premiumRequests") for result in data("result") if "usage" in result), None),
        "seconds":    round(time.time() - start),
    }
    status.update(brand.note(f"Copilot CLI finished in {run['seconds']} s with exit code {run['exit_code']}",
                             "neutral" if run["exit_code"] == 0 else "warn"))
    return run


def show_run(run, title):
    """Show the CLI's answer, then the tools it called and where each one ran."""
    assert run["exit_code"] == 0, run["error"]
    badge = brand.chip(f"{run['premium']} premium requests", "ok") if run["premium"] is not None else None
    display(brand.reply(run["answer"], title, model=run["model"], badge=badge,
                        sub=f"{run['calls']} model calls through the gateway in {run['seconds']} s", markdown=True))
    if run["tools"]:
        display(brand.table("Tools the agent called", ["Tool", "Calls", "Runs"], [
            (brand.mono(name), count, brand.chip("AI Gateway", "brand") if name.startswith("ai-gateway-") else "Your machine")
            for name, count in Counter(run["tools"]).items()]))


if copilot:
    version = subprocess.run([copilot, "--version"], capture_output=True, text=True).stdout.splitlines()[0]
    display(brand.info("Copilot CLI settings", [
        ("CLI", brand.mono(version)),
        *[(name, brand.masked(value) if "KEY" in name else brand.mono(value)) for name, value in cli_settings.items()],
    ]))
    show_run(run_copilot("Which model are you? Answer in one sentence."), "Copilot CLI · quick question")
else:
    display(brand.note("The Copilot CLI isn't on the kernel's PATH: the notebook skips the CLI steps. "
                       "Install it, restart the kernel, and rerun to try them.", "warn"))

### 5. Run an agent task with the CLI

Ask the CLI to change the project and run its tests. `copilot -p` can't ask for approval, so `--allow-tool` gives it
two permissions up front: write files, and run Python. The agent reads, plans, edits, and runs commands on your
machine, and each of its model calls goes through the gateway. Then the cell checks the agent's work itself.

In [ ]:
if copilot:
    task = "Add a multiply function to calc.py, and unittest tests for it in test_calc.py, then run the tests."
    show_run(run_copilot(task, "--allow-tool=write", "--allow-tool=shell(python:*)", "--allow-tool=shell(python3:*)"),
             "Copilot CLI · agent task")

    tests = subprocess.run([sys.executable, "-m", "unittest", "-q"], cwd=project, capture_output=True, text=True)
    summary = " · ".join(line for line in tests.stderr.splitlines() if line.startswith(("Ran", "OK", "FAILED")))
    work = [
        ("calc.py defines multiply", "def multiply" in (project / "calc.py").read_text(), ""),
        ("test_calc.py exists", (project / "test_calc.py").exists(), ""),
        ("python -m unittest passes", tests.returncode == 0, summary),
    ]
    display(brand.table("Check the agent's work", ["Check", "Result", "Detail"], [
        (label, brand.chip("Pass", "ok") if ok else brand.chip("Fail", "bad"), brand.mono(detail) if detail else "")
        for label, ok, detail in work]))
    assert all(ok for _, ok, _ in work), tests.stderr[-2000:]

### 6. Run the Copilot SDK sample

The SDK starts its own Copilot runtime and opens a session with the gateway as its `provider`: the base URL, the
key, and the Responses API. The cell runs [infra/copilot_sdk.py](infra/copilot_sdk.py) from the `infra` folder with
the kernel's Python. The agent says which model it is, and lists the folder's files with its own tools.

In [ ]:
sdk_settings = {
    "AI_GATEWAY_URL":     lab.gateway_url,
    "AI_GATEWAY_API_KEY": api_key,
    "AI_GATEWAY_MODEL":   model,
    "COPILOT_OFFLINE":    "true",
    "COPILOT_HOME":       str(copilot_home),
}


def run_sdk(*prompt, mcp_url=None):
    """Run the SDK sample with an optional prompt, and return its last message."""
    env = {**os.environ, **sdk_settings, **({"AI_GATEWAY_MCP_URL": mcp_url} if mcp_url else {})}
    status = brand.live(brand.note("Running copilot_sdk.py", "wait"))
    start = time.time()
    done = subprocess.run([sys.executable, "copilot_sdk.py", *prompt], cwd="infra", env=env,
                          capture_output=True, text=True, timeout=600)
    status.update(brand.note(f"Ran copilot_sdk.py in {round(time.time() - start)} s", "neutral"))
    assert done.returncode == 0, done.stderr[-2000:]
    return done.stdout.strip()


answer = run_sdk()
display(brand.reply(answer, "Copilot SDK · default prompt", model=model, sub="python infra/copilot_sdk.py",
                    badge=brand.chip(f"Names {model}", "ok") if model in answer else brand.chip("No model name", "warn"),
                    markdown=True))

### 7. Publish the Microsoft Learn tools

Copilot can call MCP tools through the gateway too. The cell reuses the `learn` MCP server from lab 04, or deploys
it with lab 04's [tool-server.bicep](../04-federate-mcp-tools/infra/tool-server.bicep). Then it lists the tools the
way Copilot does: `initialize`, then `tools/list`, with the key in the `api-key` header.

In [ ]:
created_server = lab.arm("GET", f"{lab.workspace_id}/toolServers/{server_name}") is None
if created_server:
    lab.deploy("../04-federate-mcp-tools/infra/tool-server.bicep", gatewayName=lab.gateway_name, serverName=server_name)
mcp_url = f"{lab.gateway_url}/default/toolservers/{server_name}/mcp"


def mcp_status():
    """Send an MCP initialize with the key, and return the HTTP status."""
    return requests.post(mcp_url, timeout=60, headers={
        "api-key": api_key, "Content-Type": "application/json", "Accept": "application/json, text/event-stream"},
        json={"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {
            "protocolVersion": "2025-03-26", "capabilities": {}, "clientInfo": {"name": "aigateway-labs", "version": "1.0"}}},
    ).status_code


assert utils.wait_for_status(mcp_status, 200) == 200, "The MCP endpoint doesn't accept the key yet. Rerun the cell."
headers, _ = utils.mcp_session(mcp_url, api_key)
tools = utils.mcp_request(mcp_url, headers, "tools/list")["result"]["tools"]
display(brand.table(f"Tools on the {server_name} MCP server", ["Tool", "Description"], [
    (brand.mono(tool["name"]), " ".join(tool.get("description", "").split())[:120]) for tool in tools],
    sub=mcp_url, badge=brand.chip(f"{len(tools)} tools", "brand")))

### 8. Give the CLI the Learn tools

`--additional-mcp-config` adds the server for one run only, so nothing lands in a configuration file. The header
`api-key: ${AI_GATEWAY_API_KEY}` reads the key from the environment when the CLI connects, so the key isn't on the
command line. `--allow-tool=ai-gateway-learn` allows the server's tools. The CLI prefixes each tool with the server's
name, for example `ai-gateway-learn-learn_microsoft_docs_search`.

In [ ]:
mcp_config = json.dumps({"mcpServers": {"ai-gateway-learn": {
    "type": "http", "url": mcp_url, "headers": {"api-key": "${AI_GATEWAY_API_KEY}"}, "tools": ["*"]}}})
question = ("Use the Microsoft Learn tools to answer: how do I run a Python function app locally with "
            "Azure Functions Core Tools? Answer in two sentences and cite the Learn page.")

if copilot:
    learn_run = run_copilot(question, "--additional-mcp-config", mcp_config, "--allow-tool=ai-gateway-learn")
    show_run(learn_run, "Copilot CLI · with the Learn tools")
    assert any(tool.startswith("ai-gateway-learn-") for tool in learn_run["tools"]), \
        "The agent didn't call the Learn tools. Rerun the cell."

### 9. Give the SDK the Learn tools

With `AI_GATEWAY_MCP_URL` set, the sample adds the server to the session's `mcp_servers`, with the key in the
`api-key` header, and approves every tool call. The answer differs between runs.

In [ ]:
answer = run_sdk(question, mcp_url=mcp_url)
cites = "learn.microsoft.com" in answer
display(brand.reply(answer, "Copilot SDK · with the Learn tools", model=model, sub=f"AI_GATEWAY_MCP_URL={mcp_url}",
                    badge=brand.chip("Cites Microsoft Learn", "ok") if cites else brand.chip("No citation", "warn"),
                    markdown=True))

### 10. Cut Copilot off by suspending the key

Every Copilot surface sends the `github-copilot` key, so one change switches them all off, on every machine,
without touching their settings. The cell suspends the key, waits until the model endpoint and the MCP endpoint
both reject it with `401`, and runs the CLI again. Then it sets the key back to `active` and checks that Copilot
works again. The key is reactivated even if a check fails.

If you set up monitoring in [lab 13](../13-monitoring/README.md), open **Monitoring** › **Models** in the portal
after a few minutes. **Model usage by caller identity** shows this notebook's calls under the `github-copilot` key,
with far more input tokens than output tokens: that's the context the agent sends with every request.

In [ ]:
key_id = f"{lab.gateway_id}/apiKeys/{key_name}"


def set_key_state(state):
    return lab.arm("PATCH", key_id, {"properties": {"state": state}})["properties"]["state"]


def model_status():
    """Send a small Responses API request with the key as a bearer token, and return the HTTP status."""
    return requests.post(f"{base_url}/responses", timeout=120, headers={"Authorization": f"Bearer {api_key}"},
                         json={"model": model, "input": "Say hi in three words", "max_output_tokens": 16}).status_code


def cli_status(run):
    return 200 if run["exit_code"] == 0 else (run["failures"] or [run["exit_code"]])[-1]


rows = []
try:
    display(brand.note(f"API key {key_name} is {set_key_state('suspended')}", "warn"))
    rows.append(("Model endpoint, key suspended", utils.wait_for_status(model_status, 401), 401))
    rows.append(("MCP endpoint, key suspended", utils.wait_for_status(mcp_status, 401), 401))
    if copilot:
        run = run_copilot("Say hi")
        rows.append(("Copilot CLI, key suspended", cli_status(run), 401, run["error"].splitlines()[0] if run["error"] else ""))
finally:
    display(brand.note(f"API key {key_name} is {set_key_state('active')}", "ok"))

rows.append(("Model endpoint, key active", utils.wait_for_status(model_status, 200), 200))
rows.append(("MCP endpoint, key active", utils.wait_for_status(mcp_status, 200), 200))
if copilot:
    run = run_copilot("Say hi in three words")
    rows.append(("Copilot CLI, key active", cli_status(run), 200, run["answer"]))

display(brand.checks("One key switches Copilot off and on", rows, sub=key_name))
assert all(row[1] == row[2] for row in rows)

### 11. Clean up resources

The cell deletes the temporary `COPILOT_HOME`, and what this notebook created: the practice project, the
`github-copilot` key, the `learn` MCP server, and the alias, the backend model, and the Foundry deployment of
`gpt-6.1-sol`. It keeps the gateway and the provider, because other labs reuse them, and anything that existed
before the notebook ran. Your own Copilot settings are untouched: the notebook added nothing to them.

In [ ]:
shutil.rmtree(copilot_home, ignore_errors=True)
if created_project:
    shutil.rmtree(project)
    display(brand.note(f"Deleted {project}", "ok"))

if created_key:
    lab.delete(f"{lab.gateway_id}/apiKeys/{key_name}")
if created_server:
    lab.delete(f"{lab.workspace_id}/toolServers/{server_name}")
if created_alias:
    lab.delete(f"{lab.workspace_id}/aliases/{model}")
    lab.delete(f"{lab.workspace_id}/modelProviders/{provider}/models/{model}")
    lab.delete(f"{lab.foundry_id}/deployments/{model}", api_version=utils.COGNITIVE_API)

kept = [name for name, created in [(key_name, created_key), (server_name, created_server), (model, created_alias),
                                   (str(project), created_project)] if not created]
if kept:
    display(brand.note(f"Kept what existed before this notebook ran: {', '.join(kept)}", "neutral"))

## Troubleshooting

- **The deployment fails with `InsufficientQuota`**: the Foundry account doesn't have 100,000 tokens per minute of
  `gpt-6.1-sol` quota in its region. Request quota, or pass a smaller `capacity`, in thousands of tokens per minute,
  to `lab.deploy`.
- **The notebook skips the CLI steps**: `shutil.which("copilot")` didn't find the CLI. Install it, check that
  `copilot --version` works in a terminal, then restart the editor so the kernel gets the new `PATH`.
- **`HTTP 401` with the right key**: a new key can take a minute to work. Rerun the cell. If the key is still
  suspended after a failed run, rerun step 10, which sets it back to `active`.
- **`HTTP 429`**: Copilot's requests carry its system prompt, its tool definitions, and the conversation, so they
  can use up the 100,000 tokens per minute. Wait a minute, then rerun the cell.
- **The agent doesn't call the Learn tools**: check that step 7 lists tools, then rerun the cell. The answer and the
  tools that the agent picks differ between runs.
- **`ModuleNotFoundError: No module named 'copilot'`**: the SDK isn't in the kernel's environment. Rerun the `%pip`
  cell, then restart the kernel.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-24-github-copilot-byok).

## Next steps

- Connect VS Code and the GitHub Copilot app to the same alias and MCP server with the [lab guide](README.md).
- Let a coding agent wire your gateway's models and tools into an app in
  [lab 17](../17-coding-agent-plugin/README.md) and its notebook [17-coding-agent-plugin.ipynb](../17-coding-agent-plugin/17-coding-agent-plugin.ipynb).
- Choose which Learn tools Copilot sees in [lab 05](../05-mcp-tool-controls/README.md) and its notebook
  [05-mcp-tool-controls.ipynb](../05-mcp-tool-controls/05-mcp-tool-controls.ipynb).
- Docs: [Use BYOK models in the GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/customize-copilot/use-byok-models)
  and [BYOK in the Copilot SDK](https://github.com/github/copilot-sdk/blob/main/docs/auth/byok.md).
- Read this lab on the labs site: [https://azure.github.io/aigateway/labs/24-github-copilot-byok/](https://azure.github.io/aigateway/labs/24-github-copilot-byok/).